In [1]:
from bs4 import BeautifulSoup
from selenium import webdriver      
from selenium.webdriver.common.by import By
import time 
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import pandas as pd

In [2]:
url = 'https://www.dges.gov.pt/guias/indcurso.asp'
browser = webdriver.Chrome(options=Options())  # Initialize a Chrome browser instance with options
browser.get(url)  # Open the URL in the browser
time.sleep(1)  

In [3]:
def scrape_current_letter(browser, course_institution_data):
    """Scrape all courses & institutions from the current letter page."""
    WebDriverWait(browser, 20).until(
        EC.presence_of_all_elements_located((By.CLASS_NAME, "box10"))
    )

    soup = BeautifulSoup(browser.page_source, "html.parser")
    all_blocks = soup.find_all(["div"], class_=["box10", "lin-curso"])
    current_course = None

    for block in all_blocks:
        classes = block.get("class", [])

        # --- Course block ---
        if "box10" in classes:
            name_tag = block.find("div", class_="lin-area-c2")
            if name_tag:
                current_course = name_tag.text.strip()
                print(f"\n📘 Course: {current_course}")

        # --- Institution block ---
        elif "lin-curso" in classes and current_course:
            link_tag = block.find("a")
            if not link_tag:
                continue

            institution = link_tag.text.strip()
            href = link_tag.get("href")

            if any(kw in institution for kw in ["Universidade", "Instituto", "Escola", "Politécnico"]):
                print(f"🏫 Institution: {institution}")

                try:
                    # Click the institution
                    inst_element = WebDriverWait(browser, 10).until(
                        EC.element_to_be_clickable((By.XPATH, f"//a[@href='{href}']"))
                    )
                    browser.execute_script("arguments[0].scrollIntoView(true);", inst_element)
                    time.sleep(0.3)
                    inst_element.click()

                    # Wait for detail page
                    WebDriverWait(browser, 15).until(
                        EC.presence_of_all_elements_located((By.CLASS_NAME, "inside2"))
                    )
                    time.sleep(1)

                    # Parse the detail page
                    detail_soup = BeautifulSoup(browser.page_source, "html.parser")

                    # --- Extract Google Maps link ---
                    google_map = ""
                    inside_block = detail_soup.find("div", class_="inside2")
                    if inside_block:
                        map_link = inside_block.find("a", href=True, string=lambda t: t and "Mapa" in t)
                        if not map_link:
                            map_span = inside_block.find("span", class_="vislink", string=lambda t: "Mapa" in t)
                            if map_span and map_span.parent.name == "a":
                                map_link = map_span.parent
                        if map_link:
                            google_map = map_link["href"].strip()

                    print(f"🗺️ Google Maps link: {google_map if google_map else 'Not found'}")

                except Exception as e:
                    print(f"⚠️ Error scraping {institution}: {e}")
                    google_map = ""

                # Go back to the list page
                browser.back()
                time.sleep(1)
                WebDriverWait(browser, 20).until(
                    EC.presence_of_all_elements_located((By.CLASS_NAME, "box10"))
                )

                # Store
                course_institution_data.append((current_course, institution, href, google_map))

    return course_institution_data


# --- Step 1: Scrape letter A (already selected) ---
print("\n🔤 Scraping letter: A (default)")
course_institution_data = []
course_institution_data = scrape_current_letter(browser, course_institution_data)

# --- Step 2: Click and scrape all other letters ---
letters = browser.find_elements(By.CSS_SELECTOR, "div.noprint a")
letter_links = [(a.text.strip(), a.get_attribute("href")) for a in letters if a.text.strip()]

for letter, link in letter_links:
    print(f"\n🔤 Scraping letter: {letter}")
    browser.get(link)
    time.sleep(1.5)
    course_institution_data = scrape_current_letter(browser, course_institution_data)

# --- Save results ---
df = pd.DataFrame(course_institution_data, columns=["Course", "Institution", "Link", "GoogleMaps"])


🔤 Scraping letter: A (default)

📘 Course: Acupuntura
🏫 Institution: Instituto Politécnico da Lusofonia - Escola Superior de Saúde Ribeiro Sanches
🗺️ Google Maps link: http://maps.google.pt/maps?z=20&t=k&q=loc:38.74674+-9.100184

📘 Course: Administração e Gestão de Empresas
🏫 Institution: Universidade Católica Portuguesa - Faculdade de Ciências Económicas e Empresariais
🗺️ Google Maps link: Not found

📘 Course: Administração e Gestão de Empresas - Licenciatura Internacional
🏫 Institution: Universidade Católica Portuguesa - Faculdade de Ciências Económicas e Empresariais
🗺️ Google Maps link: Not found

📘 Course: Administração Pública
🏫 Institution: Universidade de Aveiro
🗺️ Google Maps link: http://maps.google.pt/maps?z=20&t=k&q=loc:40.63118+-8.657394
🏫 Institution: Universidade de Coimbra - Faculdade de Direito
🗺️ Google Maps link: http://maps.google.pt/maps?z=20&t=k&q=loc:40.20785+-8.425645
🏫 Institution: Universidade de Lisboa - Instituto Superior de Ciências Sociais e Políticas
🗺️ G

In [4]:
df

,Course,Institution,Link,GoogleMaps
0,Acupuntura,Instituto Politécnico da Lusofonia - Escola Su...,detcursopi.asp?codc=L160&code=4614,http://maps.google.pt/maps?z=20&t=k&q=loc:38.7...
1,Administração e Gestão de Empresas,Universidade Católica Portuguesa - Faculdade d...,detcursopi.asp?codc=9059&code=2270,
2,Administração e Gestão de Empresas - Licenciat...,Universidade Católica Portuguesa - Faculdade d...,detcursopi.asp?codc=L149&code=2270,
3,Administração Pública,Universidade de Aveiro,detcursopi.asp?codc=9002&code=0300,http://maps.google.pt/maps?z=20&t=k&q=loc:40.6...
4,Administração Pública,Universidade de Coimbra - Faculdade de Direito,detcursopi.asp?codc=9002&code=0502,http://maps.google.pt/maps?z=20&t=k&q=loc:40.2...
...,...,...,...,...
1567,Turismo em Espaços Rurais e Naturais,Instituto Politécnico de Coimbra - Escola Supe...,detcursopi.asp?codc=L178&code=3061,http://maps.google.pt/maps?z=20&t=k&q=loc:40.2...
1568,Turismo e Lazer,Instituto Politécnico da Guarda - Escola Super...,detcursopi.asp?codc=9255&code=3095,http://maps.google.pt/maps?z=20&t=k&q=loc:40.4...
1569,"Turismo, Território e Patrimónios",Universidade de Coimbra - Faculdade de Letras,detcursopi.asp?codc=L109&code=0505,http://maps.google.pt/maps?z=20&t=k&q=loc:40.2...
1570,Zootecnia,Instituto Politécnico de Coimbra - Escola Supe...,detcursopi.asp?codc=L003&code=3061,http://maps.google.pt/maps?z=20&t=k&q=loc:40.2...
